# Analyse d'incertitude calibrée

Ce notebook ajoute une étape méthodologique au projet FreshOrRotten.

Le modèle CNN prédit toujours `fresh` ou `rotten`. Mais dans un usage réel, certaines images peuvent être trop éloignées des données vues pendant l'entraînement. L'objectif est donc de tester une règle `uncertain` qui rejette les prédictions peu fiables.

## Objectif

On compare deux systèmes :

- baseline CNN : le modèle prédit toujours `fresh` ou `rotten` ;
- baseline CNN + incertitude : le modèle prédit seulement si son score de confiance dépasse un seuil calibré sur validation.

Le seuil n'est pas choisi à la main. Il est calibré avec le `validation_set`, puis évalué sur le `test_set` standard et le `test_set` avec catégories non vues.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

reports_dir = project_root / "reports"
project_root

## Lancer l'analyse

Cette cellule utilise les modèles déjà entraînés et les splits déjà sauvegardés.

Elle ne réentraîne pas le CNN.

In [ ]:
command = [sys.executable, str(project_root / "src" / "uncertainty.py"), "--protocol", "both"]
subprocess.run(command, cwd=project_root, check=True)

## Résultats globaux

`coverage` indique la part d'images acceptées par la règle.

`uncertain_rate` indique la part d'images rejetées comme incertaines.

In [ ]:
metrics_path = reports_dir / "uncertainty_metrics.csv"
uncertainty_metrics = pd.read_csv(metrics_path)
uncertainty_metrics

In [ ]:
plot_columns = ["baseline_accuracy", "accepted_accuracy", "coverage", "uncertain_rate"]
ax = uncertainty_metrics.set_index("protocol")[plot_columns].plot(kind="bar", figsize=(9, 5))
ax.set_ylim(0, 1)
ax.set_title("Baseline vs prédictions acceptées")
ax.set_ylabel("Valeur")
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Résultats par product_type

Cette partie permet de voir si la règle `uncertain` rejette davantage certaines catégories.

In [ ]:
product_type_path = reports_dir / "uncertainty_by_product_type.csv"
uncertainty_by_product_type = pd.read_csv(product_type_path)
uncertainty_by_product_type

In [ ]:
unseen_product_metrics = uncertainty_by_product_type[
    uncertainty_by_product_type["protocol"] == "unseen"
].copy()

ax = unseen_product_metrics.set_index("product_type")[["baseline_accuracy", "accepted_accuracy", "uncertain_rate"]].plot(
    kind="bar",
    figsize=(9, 5),
)
ax.set_ylim(0, 1)
ax.set_title("Incertitude sur les catégories non vues")
ax.set_ylabel("Valeur")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Points à analyser

À compléter après exécution :

- est-ce que `accepted_accuracy` est supérieure à `baseline_accuracy` ?
- combien d'images deviennent `uncertain` ?
- est-ce que le taux `uncertain` augmente sur les catégories non vues ?
- est-ce que les images rejetées contiennent beaucoup d'erreurs du modèle ?

Si la règle rejette plus d'images difficiles sur le `unseen_category_split`, cela soutient l'idée que le modèle doit exprimer son incertitude quand il rencontre des produits moins proches de son entraînement.